In [ ]:
%pip install ipywidgets
import io, string, warnings, zlib, base64
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from scipy import ndimage
from ipywidgets import IntSlider, FloatSlider, Dropdown, VBox, HBox, Label, Layout, Image as ImageWidget
from IPython.display import display, Image, clear_output
warnings.filterwarnings('ignore')

slider_layout = Layout(width="420px")
label_style = {"description_width": "120px"}


# ---------------------------------------------------------------------------
# Display and plotting helpers
# ---------------------------------------------------------------------------

def figure_png(fig):
    """Render a matplotlib figure to PNG bytes (and close it). Used to update the pictures in the widgets."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    return buf.read()


def show_figure(fig):
    """Render a matplotlib figure to a PNG and display it in the notebook."""
    display(Image(data=figure_png(fig)))


def _display_range(image, num_std=3):
    """Grey-scale limits for showing an image: its full range, clipped at +-3 standard deviations,
    so that a few extreme noise pixels do not turn the picture into black and white speckle."""
    mean, std = image.mean(), image.std()
    return max(image.min(), mean - num_std * std), min(image.max(), mean + num_std * std)


def imshow(*images, titles=None, size=3.0):
    """Show one or more images side by side (gray scale). Example: imshow(im1, im2, titles=['a', 'b'])"""
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for i, (ax, image) in enumerate(zip(axes[0], images)):
        low, high = _display_range(image)
        ax.imshow(image, cmap='gray', origin='lower', vmin=low, vmax=high)
        ax.axis('off')
        if titles is not None:
            ax.set_title(titles[i], fontsize=9)
    plt.tight_layout()
    show_figure(fig)


def show_stack(images, n=24, columns=8, angles=None):
    """Show the first n images of a stack as small thumbnails (with their angles as titles, if given)."""
    n = min(n, len(images))
    rows = int(np.ceil(n / columns))
    tile = 1.35
    fig, axes = plt.subplots(rows, columns, figsize=(tile * columns, tile * rows * (1.15 if angles is not None else 1.0)),
                             squeeze=False)
    for i, ax in enumerate(axes.ravel()):
        ax.axis('off')
        if i < n:
            low, high = _display_range(images[i])
            ax.imshow(images[i], cmap='gray', origin='lower', vmin=low, vmax=high)
            if angles is not None:
                ax.set_title(f"{angles[i]:.0f}°", fontsize=8)
    plt.tight_layout()
    show_figure(fig)


def plot_data(data, xlabel, ylabel, title):
    """Line plot. `data` is a pair (x, y). Example: plot_data((angles, scores), 'angle (°)', 'score', 'title')"""
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(data[0], data[1])
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.grid(alpha=0.3)
    show_figure(fig)


def plot_histogram(data, bins, xlabel, ylabel, title):
    """Histogram of a list/array of values."""
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.hist(data, bins=bins)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.grid(alpha=0.3)
    show_figure(fig)


def plot_scatter(data, xlabel, ylabel, title):
    """Scatter plot. `data` is a pair (x, y)."""
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(data[0], data[1], s=4)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title)
    ax.grid(alpha=0.3)
    show_figure(fig)


def best_rotation_corr(truth, reconstruction, step=5):
    """Correlation between truth and reconstruction, maximised over a global rotation of the reconstruction.

    A reconstruction can only be determined up to a global rotation (nothing in the images tells you
    which way is 'up'), so this is the fair way to compare when the reference was not oriented like the truth.
    """
    if np.std(reconstruction) == 0:
        return 0.0
    return max(np.corrcoef(np.ravel(truth), np.ravel(rotate_image(reconstruction, a)))[0, 1]
               for a in range(0, 360, step))


def compare_reconstruction(truth, reconstruction, title="Reconstruction", allow_rotation=False):
    """Show the true object next to a reconstruction and report their correlation coefficient.

    With allow_rotation=True the correlation is maximised over a global rotation of the reconstruction.
    """
    if allow_rotation:
        r = best_rotation_corr(truth, reconstruction)
    elif np.std(reconstruction) == 0:
        r = 0.0
    else:
        r = np.corrcoef(np.ravel(truth), np.ravel(reconstruction))[0, 1]
    imshow(truth, reconstruction, titles=["True object", f"{title} (r = {r:.2f})"], size=3.2)


# ---------------------------------------------------------------------------
# Making objects
# ---------------------------------------------------------------------------

def make_letter(text, size=64):
    """Render a single character as a binary 2D image (letter = 1, background = 0) of size x size pixels."""
    fig, ax = plt.subplots(figsize=(2, 2))
    ax.axis('off')
    fig.text(0.23, 0.26, text, fontsize=100)
    fig.canvas.draw()
    raw = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    width, height = fig.canvas.get_width_height()
    gray = raw.reshape(height, width, 4)[:, :, :3].mean(2)
    plt.close(fig)
    gray = (gray - gray.min()) / (gray.max() - gray.min() + 1e-10)
    binary = (~gray.astype(bool))[::-1].astype(float)
    return ndimage.zoom(binary, size / binary.shape[0])


def low_pass_filter_image(image, cutoff=5):
    """Remove details smaller than `cutoff` pixels (smooth tanh edge in Fourier space)."""
    ky = np.fft.fftfreq(image.shape[0])
    kx = np.fft.fftfreq(image.shape[1])
    k = np.sqrt(kx[None, :] ** 2 + ky[:, None] ** 2)        # cycles per pixel
    k_cut = 1.0 / cutoff
    mask = 0.5 * (1 - np.tanh((k - k_cut) / (0.15 * k_cut)))
    return np.fft.ifft2(np.fft.fft2(image) * mask).real


# ---------------------------------------------------------------------------
# Image operations and similarity measures
# ---------------------------------------------------------------------------

def rotate_image(image, angle):
    """Rotate an image by `angle` degrees (counter-clockwise), keeping the image size."""
    return ndimage.rotate(image, angle, reshape=False)


def translate_image(image, dx, dy):
    """Shift an image by dx pixels along x and dy pixels along y (periodic boundaries)."""
    return np.roll(np.roll(image, dx, axis=1), dy, axis=0)


def SQD(image1, image2):
    """Sum of squared differences: small = similar."""
    return ((image1 - image2) ** 2).sum()


def corr(image1, image2):
    """Cross-correlation of two images: large = similar."""
    return (image1 * image2).mean()


def log_likelihood(image, reference, sigma):
    """Log-likelihood of an image given a reference and the noise level sigma (up to a constant)."""
    return -0.5 * np.sum((image - reference) ** 2) / sigma ** 2


# ---------------------------------------------------------------------------
# Simulating particle images:   X = R(theta) A + sigma G
# ---------------------------------------------------------------------------

def simulate_image(image, angle, sigma=1, rng=None):
    """One noisy, rotated copy of `image`."""
    rng = rng if rng is not None else np.random.default_rng()
    return rotate_image(image, angle) + sigma * rng.standard_normal(image.shape)


def simulate_images(image, sigma, N, seed=None):
    """N noisy copies of `image` at random angles. Returns (true_angles, images_stack)."""
    rng = np.random.default_rng(seed)
    angles = rng.uniform(0, 360, N)
    stack = np.array([simulate_image(image, a, sigma, rng) for a in angles])
    return angles, stack


def simulate_interactive(image):
    """Explore the image formation model: rotate the object and add noise."""
    angle_slider = IntSlider(value=30, min=0, max=359, step=1, description="Angle (°)",
                             style=label_style, layout=slider_layout)
    sigma_slider = FloatSlider(value=1.0, min=0.0, max=8.0, step=0.5, description="Noise σ",
                               style=label_style, layout=slider_layout)
    picture = ImageWidget(format='png')

    def update_plot(_=None):
        rotated = rotate_image(image, angle_slider.value)
        noisy = rotated + sigma_slider.value * np.random.standard_normal(image.shape)
        vmax = 1.2 * np.abs(image).max()
        vmax_noisy = max(vmax, 3 * sigma_slider.value)       # keeps the noise from saturating into black and white
        fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.4))
        axes[0].imshow(image, cmap='gray', origin='lower', vmin=-vmax, vmax=vmax)
        axes[0].set_title("Object A", fontsize=9)
        axes[1].imshow(rotated, cmap='gray', origin='lower', vmin=-vmax, vmax=vmax)
        axes[1].set_title(f"Rotated by {angle_slider.value}°", fontsize=9)
        axes[2].imshow(noisy, cmap='gray', origin='lower', vmin=-vmax_noisy, vmax=vmax_noisy)
        axes[2].set_title(f"Observed image X (σ = {sigma_slider.value:.1f})", fontsize=9)
        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        picture.value = figure_png(fig)

    for widget in [angle_slider, sigma_slider]:
        widget.observe(update_plot, names='value')
    display(VBox([angle_slider, sigma_slider, picture]))
    update_plot()


def rotate_viewer(image):
    """Show a reconstruction and turn it with a slider until you recognise it.

    The orientation of a reconstruction is arbitrary: nothing in the images says which way is 'up'.
    """
    angle_slider = IntSlider(value=0, min=-180, max=180, step=5, description="Rotate by (°)",
                             style=label_style, layout=slider_layout)
    picture = ImageWidget(format='png')

    def update_plot(_=None):
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(rotate_image(image, angle_slider.value), cmap='gray', origin='lower')
        ax.axis('off')
        picture.value = figure_png(fig)

    angle_slider.observe(update_plot, names='value')
    display(VBox([angle_slider, picture]))
    update_plot()


# ---------------------------------------------------------------------------
# Unknown test data sets (the objects are hidden; every set has its own defect)
# ---------------------------------------------------------------------------
# The hidden objects are stored as small embedded bitmaps (8 bit, zlib + base64): (image size, data)
_HIDDEN_OBJECTS = {
    1: (128, 'eNrtWmtMm+cVrm8YsB2T2CEQLiYQCMVdIBDiBApJKCWKabNgN2lCEy6FGoxtHNsk2GAbAwZjG2yM77c0XZZsWVW1W6tNW7dqVSutWrtJW9utnaZ1WlW1TXrR1EvSpp32fp/v5jP+TEx++fmD+OPnfM8573nPOe+5554MMsgggwwyyCCDDBIDkwh3hxuLw+GRgMNhN9YGmBtPIOaQyGRKPMhkUg6RgN8wG4LcueQ8ekFxKWM1SosL6Hnk3KAN6ScPcBeW7mTW729uaWmNRUtL8/565s7SwoANaTUhQE6lQdwNTW1sbndPX19/LPr6erq57LamBsgGGjVgQto+PYdC215eU9/U1sk9MyiQKZRq9VQs1GqlQiYYPMPtbGuqrynfTqPkpEMEwI4nkqj5pdUNrUcBt1A2oZkzmJYtFstKNMD/yybDnGZCJgQ2HG1tqC7Np5KI+Du0ALBnQ59+34EObi9fCnOv2JwuDxJcTtsKbIOU38vtOHAfJEI2/g68gMHgski0kuqGg52neFLlLMzt9nh9fmT4vB43bMOsUso71XmwobqERsrCrVcC6OOphVWN7dxewQXNgtnqCHJfTISgDQ6reUFzQdDLbW+sKqSuVwIMlkCiMXYf4g5IlFqjxeFemzvGBrfDYtQqJQPcQ7sZNBJhHQYA7YmUbVUs9lmxCny60+NFwR2xwetxAhFU4rNsVtU2CjFlH0Da5xUxW7m88Tmz3QV9+sVUAIngspvnxnncVmZRXqo+gLUvq2vvHlUbrBD7xdQBWWA1qEe72+vKUvQBoCfnV7I6+2SzJocnwh4f8Ekt8DhMs7K+TlZlPjkVAzD43PzqFs6QXGdxef2RuALBHQWvL2lA+r0ui04+xGmpzs/Fo+bH4LJplS0nhWqj1R38+ODhdtisYdgccC5YOzL8PrfVqBaebKmkZeMwaNXPojJYHKEmrD0Uzy6H1WJeNOj1CzD0esOi2WK1B/PR2j7QCDksBjULpQcweFLhbjZPbXIGtQe/4bQtLxl02mm1ShmESj2t1ekXzSuQCWta4HWa1Dz27kISOg9A6lcdOjtusIfovW67xTg/rZ6Un5dJJUFIZeflk8CGBdiEtdKD32s3jJ89VIXSAxgCpWQfVzxn9YToXVaTbnrygkwiFgkFAsEIAPgjFIklsvMKpQaYsGxzehIfUr/HOifm7iuhEDCoPp9+b8eg2uzyheRbMWpV49JR4QifPzw8FMTwMJ8/AmyQAhOm540WuzuxBD6XWT3YcS8djQAYPKWEdVKmD6rv97lWDDMTY2IhH6bmhQEbwR8RABMuTE7rTNbEEgAP6GUnWSUUFBGAJW7Z9eDg1LI7qL7HtjijkIpGhqO5o2wAJghHpeMqrWEZkiCBAe7lqcEHd20hYpPLn7Ot7vi5hcjnL88rZSLw8bwEgEwQiscmNEACVwIDgAAL547XbctJ6gAMjlTMOi1fCnvfsTQzLh4ZSkgfsGBEJAESGFecCQzwuZbkp1nFpOT8eHJpc48ySn7j1JhweC16yIJhvgBIMGNYSaAAcICyp7mUjEfBz2jpU1s8YX69SioY5iUDJIFUMRPJ2PFH0KLua2Gg4aesix+SQCSd0JrsnjX4Kej4+6ei+NHoH4gCvmhMpbMghgDgn+pHzR/1/XD8jfJR8EMuEI9PG20e/x19f6z/ofM3KUUlADBAKJ2cX3YhREAK/o+Jf9gB0+AAojOAPzo+s+RAcAD6+I89/1D6t+hUYyL+0BAqAcamEB2A/vzH5j9IALtJOyFdKwNGMCyQqvQI/OjzX3z+D1RQwRtg/fyo83/8/Qff/4EbUMBPngaR9U/l/oPu/5qo+z9SAUgSXIJRGQAcQKT4S+X+D9Q/j5ybt4V/JlABaUJFwFoZUKZEOH8p1T9w/bfrgX6lKSwAXAEuG0JF0PAQcikQuAEWV8kPcpgxhfovIMAJmd7hi66inYEiUCoWCUaQSyFh8AaM/3yQwqaG2LVo61+4/t7TBY6gL6ajDRTBSsV5YAJSKQgXQQgVgN9r00lO7C9DX/+v/v5wE7C4oNUAE5BK4UARuPr+B2cPBF/N1hyU/Q/IQPTqWP9Hmnr7ihmYgNwKBIvg1alvUX6mpWIzEaX3oe5r/wlJVPxHNfWgB4RM0K9uhQxLCZoAwK84e7CKnotuBhDovoamzHGfH92G2pFaQYcLuQkC+muGH27cQQMGoO++jI5EpSzchiO2wsj9B9R+yQceYu2ko2nB47uv6LlD9IgphVGA321dUPCOH6hAkQBA8G2Nzb5BzV1xs7cURiHQDWZQ8o/vY+Qli0Eo+MqaTo2Fbx94mOa0WcwmsyU8e0x9EuSxLyp5nXXFm7JwWOwajyZQ8NV2Dk9b4Ns3QG41G+amJhWTU9qE88/kFnjsRkV/x+4iag4xKwb4aEWgu2/X4R45FHx++LzbzAatckwwcKb7zOMjMuT5rxfFdA4YYJD3H9lTXpi/lR4F2qbsSFAGBi+PnJsDTYQvNMQcG+k7wT7cfKD5MDvB/NsKREk+ofO7zVN87gMHGhuiUF9bAfJi3NFXm5weD6Q7NMQF5Af3MisYJaUVCeb/mpn5RasriQFPXrp89eoli0bKf2JwIIL+x9p2bSbEHX291emwQrrDQ+y9zPLt9DwKmUKlI79/DAyJJ/VWTxL+y9d+8eJLv3726iVwdMPwukyDTYXEED9Qfx9XNLNkWYZ1h4f4FUV0KimbgMfjE7z/HGw7+uiIGjFbRn/+j5958Y9vv/O3v/75T29E8PofnlcdKc0OfX5uwX1HHh/X6uenIN1DjxjZ8DsKBvH9q6ysbEfVnvYexaIzCf+V537/5vs3Pv30kxtBXL9+/eMP//PqfCeDGMp81PImLn98UiF5Iqh76BEHs8b7H3Xrzvsfi/QLifiv/vyVdz7+8tY3Qdy6dfPmV198/tFLmo5iQqjwpjMPnxwY5vWEdEd6R4t//yQQqYzmMxNmtz8Z/6vv3vj6u+8D+O72t7e++u8nH/zjJ6KmrcH4x+Xk/+DgQ10/PBKje7L3XyyBwri/R7XsQcH/yc3v/gfw/fe3v7n5xWcf/uut116Y7rp3EybEv7WqYT9rT82OVB7P4vrl5PxB8vfeeu03T3vUJ+oLiOHGJ6+ovLy0EFn3dPDf+Pr27W8j5AvjT7B3byfhwr+UQ6Vt3pTi4ynE3xqZV6zJf/3Lm19+/tF7bwfJH3mQVRFVE0Avjdkpv9yuyQ+VCT4oM8Lx//cPP/v0g3dffzFEziyDX4aiLl8sNuXX8zX4wbXrcthBWQYsAPwvv/nv9//5xi8vGYPkBZvJxLiScB1P94n9D3VtSwu6hSVQk1+8dOXZ373+1l9eedokOd0RIAdJNQ0v1HHzmpiuy6xVSEYlCq3Z6bt05Zlfvfzqb68Z+UfrK0Lk6XgfD81rPAg977S4l3OM0yueNrsuXr723AvPX7NKj9WCGiQLn7b1gOC8xuRa3fNqxafa9tbubTsl1lq9T1356c+uumf6W3dQibh07ibgcgr2cKRxzRo88FH0tjFLCkqYbb0TS+6LT13+kc90/kRDYS4urcsZWCKt5ih/1hrXLvgcCxLO3pI8cl7JXo4EmPfkk3773MjRGjqKGUxKAhCoO1p7J+MLAK9dN3qstoCcTS6oPTaqA700KMCUva3lVEJ6d1NA2bC9gSPR2WMFiP3+BYcPGoFJOA3b0yw/7IDq9gGVyRlTAsb4XwHKA6/TpBpor6alWX74BBY3dolmLW4fcvw/KtauuL0uy6yoq7GYjE/3dhIGS9xS2dot08XWoNHnHxTUbqtO1t1auYWITft2FCjbtzHb++SGmNF/VP4zOdxu0H70tTO3kfDpX86CaqCiOjZPuWiPnj+E8/+K0+22gf6PXVdEIWA3YDkMeABU7sf5SkPM60vw/gPsrmD/SyViN2Y3DXQuFQeO8+TQ60f0uoLPC+BxWnRylP3/ukMgl16xjw3tSwSWRaJWglw2aAeCva+CnovfsO1ALD6Xxqhr7xapdGZoEhSE2+WwmnQqUXd7HQPd/OcOFrWoRczWrgHJ5Kwe7o8t8E6YfnZSMtDVyiyiZuOxG7ocCe0LVTa2c3r5UnmgP1arlXIpv5fT3lgZW+ltkAQE0pZiaE+OczrQH/f19ZzmQG1k8RYSAbvh26lQAU2mbd9Rs4cV6I9bWpoD7cyqQnOj9lOBBblUWqQ/Li0ugHYh8di7spsb2hMN98eBnVjc3WIPm4CL3Qm+5y7jbu9EZ5BBBhlkkEEGGWSQKv4PPkrrzg=='),
    2: (112, 'eNrtWE2IHEUU7uqerp6ZlTUogkaWiD/gDxpBVvfgwV+IFw0SBY1g9CAaI4YVES/LRkFk1YhCDgajcf1BjOiqxMuKB9GARo2JggHBhBgl0WQXYXa6unumyq73unf6dzLdPQpCvdN01+v6Xr169b2vRtOUKVOmTJkyZcqUKVOm7N8z8l9gWDe5HEyIPI+rO+DQbe+k1QHpGqcDxkWei0AHr7WtPoQVfuGvTBpv/5aTUH8MXcQwMhpMJTjbm4lX37IUugwVT05nZOE9ysKI7BOV0ayn2CnwegFx52BlvOZjdgSvRvonQBi14aSzK8N3s/ZHvwMcvCAikwwDr3PcR+Qezxiv7XKkxwEJyL2qFWPsbXNZmvsR91KaCt/8gMnTEMRVGe9zLBetiXhmmuPke/f+IO98Na3GmG54sP6UCftZy8bz7jI3It6EWQ0P07RP037xsg/0I+Bxfv0qDOydavmcgEmcGVPbCb+aKY/DGIe+aigUcxJmc6dosE8brKTHMTdA8YulOt6iy3ESxHPubMQKlNRGAeRcv0qEJ13Z5vI9gtBzekmy3rR5KnxCcYMvpKSGv9g9I6VPPLEIzOFe489B51s8I13C77Uek+deG8IGAo8J97Y6QabhrTkr5RHy3H7GK+PJGcLGLiezX6NxsdHp9Y2DjqQi++PSJ5DeCMsL8IxpNxU+vRnxdHjADXy/dIugtyPrb8Snh9N42g+SXH6NUjtfLJ/Ot2B9YgwiNsbh4dZU93C+1PHhCLqX141BxZ1R6z26B2LNCFa0AwHMHcjtY9V6n7cQeYxLBmuKRVJMtzPoykfKSpeX2oAgwmK8GOpd79VDc9KObSmKmMNlpcsUShesPqmjeVIywBHZ0kjkf6SkqAgJQ4+RiUh4LL2yzJg/ou7VaHk8bu82o+QVlQy175ZiFEfvkyeGlxQV+lcghNj8sug04ETzK8PwjXkWw7PudVDLlcPbA1+LUS2BN25Gm3/ncZLagdUlKI1oi90E3xNyQtLXp2FAKF3W0pRYvb6EaNIDejoafflTlNICjyuWzwdpnIbfPFOm+Z2N9DsdjXW+E12x8BtC9FZI6EUVeqALZG0/bSVLlm0KDsCCF8eTH5UXMbaX0T4B726QDMYKGB+JHbYW4LWni4oYQi/ISk19W3v5XW0MxlfG8Cbhnf3QSGHpsgLJ+rJmdDq6K3LCN8h2Ee92ulF6A/FO3rnBIsmE8jZSju/B3XZWUZfGS1MFbOB75rJHgprJt0A57rsFRYV5rYhIl4i9LKvWXxSpr3XjZI54R+X9TTi7jYLSZT1Kl63JnviAg5KBNCd8vO4fKfGAF5zCPXcGD9JZibzUH2ShRAHp8pmR18QKtqTgs9FaxgB3/grVzEdJvPpzdmk87v6eOcDdQ5r5ZPZ9kL6IIubvYtJlFs6Zdyg9tBn+F6D1GScTj6zESI8Xwms8DzwinkgX7uUwcLr2tb/B9mQzdyfOLCRicnc9YLHrwKP9bAZPLuKnJh0KnqZ5HeCBPBqx1rnFRcUn8H+S/WGWLsB7bLebjUfq651spuhn34PQau1J45Fa7/+y7tY+zCsuGZjSiHYsv20aPbzOGtqH6W8ZWMQYej+WJ9+w8B/fcTOrk1kYzKaB6yVYApu0MvH2OVzklwQxxwr3JCgJ7mTjaaPB6nJ1dBt0TQE81rIZY2KVnhOO648yh+fdE07Kz+2FF6wBe995s7Nv+DaX5zAHw7NvN3Ka3PbX5fir6wYtUELRcpvjKcZNU46aBVuuMmXKlClTpkyZMmX/X/sHUt5m3Q=='),
    3: (96, 'eNrtmL1KxEAQgHc3uZDzFE8QRD1EsDi08Ae0EfwBO/HARr0nEAux8BHstRBBWx/EStAnsLKyEawOBS/Z/GzMZRNNItfNaDNfE24D3+wkt7OzYYwgCIIgCILIEL9GLDMGLkBUZuug3d6Lw1SQ/I7rujKKTo+YieJX+iK94BDOr4qkoUIwv1tApln4rxsWjN+ems7RfI5kmgLQ/BnjeeKfgYIJ0NcSJu/FXRFIfsMDSaC/5C1JoDuJ5Ocz+gHZHGn+DOQv9I/+ax/VL3Zd3Pl//sXzl/OofiXHkfxi0VGo67eV1FAHyW/WOsn0mxzFb+/rvXLUgpl/xcoRtw7HTm/44wlofzk7v8hx6UZ6aZ1A7b/SK5CMBS9Lgwj9yTfLTHA8f9zB4fRvGZ0HxsH8fp6sdfBvLSg/E+YPBmN6bTk3JpS/xNB9oMc5jp8P3IWw/Vs5gKVvjHCs+nnVS0DuGFj+Rq88h49o/eeYVKj9LbZ/AtmvzzUtA9OvnG0sP/T5qNCaVDL9O4p/YS0r0psCxK/KZ/j0OldF3B+jaNbmeH6vu15F23/j2rPKBAMg8mSe9FDaYDB2xur14QI1Xf7pyxtBEARBEJh8AQy0Ofs='),
}
_UNKNOWN_DATASETS = {
    1: dict(sigma=4.0, N=600, max_shift=0, junk=0.0, seed=11),
    2: dict(sigma=3.0, N=600, max_shift=7, junk=0.0, seed=12),
    3: dict(sigma=4.0, N=600, max_shift=0, junk=0.3, seed=13),
}


def _hidden_object(choice):
    size, data = _HIDDEN_OBJECTS[choice]
    return np.frombuffer(zlib.decompress(base64.b64decode(data)), dtype=np.uint8).reshape(size, size).astype(float) / 255


def load_unknown_dataset(choice):
    """Return a stack of images of an unknown object. Only the images are given, not the angles."""
    cfg = _UNKNOWN_DATASETS[choice]
    rng = np.random.default_rng(cfg['seed'])
    obj = _hidden_object(choice)
    stack = []
    for _ in range(cfg['N']):
        if rng.random() < cfg['junk']:
            clean = np.zeros_like(obj)                               # an image without a particle
        else:
            clean = rotate_image(obj, rng.uniform(0, 360))
            if cfg['max_shift']:
                clean = translate_image(clean, *rng.integers(-cfg['max_shift'], cfg['max_shift'] + 1, 2))
        stack.append(clean + cfg['sigma'] * rng.standard_normal(obj.shape))
    return np.array(stack, dtype=np.float32)

print("Setup complete.")

# Bonus exercise 1: Dealing with translation

*Based on the practical by Maarten Joosten, Stefan Huber, Alok Bharadwaj and Arjen Jakobi (High-Resolution Imaging, NB4020), adapted to run in the browser.*

So far the particle was perfectly centred. In practice, particles are picked from a micrograph and extracted in a box, so the particle sits at an unknown offset:

$$X_i = T^{\theta_i, t_i} A + \sigma G_i \tag{3}$$

where $T$ rotates by $\theta_i$ and shifts by $t_i = (dx, dy)$. Now we have to estimate the angle *and* the shift of every image. We assume integer pixel shifts.

In [ ]:
im = make_letter("R", size=96)

# what translated images look like
fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for ax, dy in zip(axes, range(-16, 16, 3)):
    ax.imshow(translate_image(im, 0, dy), cmap="gray", interpolation="none", origin="lower")
    ax.axis("off")
    ax.set_title(f"dy = {dy}", fontsize=8)
show_figure(fig)

### A dataset with rotation and translation

In [ ]:
N = 200; sigma = 1
true_angles = np.random.uniform(0, 360, N)             # the true angles
true_translations = np.random.randint(-8, 9, (N, 2))  # the true (dx, dy), an N x 2 array
images_stack = []
for angle, (tx, ty) in zip(true_angles, true_translations):
    image = translate_image(rotate_image(im, angle), tx, ty)
    images_stack.append(image + sigma * np.random.randn(*im.shape))
images_stack = np.array(images_stack)
show_stack(images_stack, n=16)

### Reconstruction without translation

First see how far your rotation-only algorithm from the main notebook gets.

In [ ]:
# YOUR SOLUTION HERE: paste your rotation-only algorithm here
im_reconstruct_rotation_only = np.zeros(im.shape)

compare_reconstruction(im, im_reconstruct_rotation_only, 'Rotation only')

### Task: estimate the translation as well

Search over rotations *and* translations. The number of combinations grows quickly (72 angles × 289 shifts = 20,808 comparisons per image, even with 1-pixel steps), so use coarser steps, and reshape images and references to 2D arrays so that all scores follow from one matrix product (`images_2d @ references_2d.T`).

```{admonition} To do
:class: note
1. Build a stack of references that are rotated and translated (for example 10° steps and shifts from −8 to 8 pixels in steps of 2).
2. For every image, find the best (angle, dx, dy).
3. Undo the estimated rotation and translation of every image and average them.
4. Compare with the rotation-only result, and the estimated with the true translations.
```

In [ ]:
# YOUR SOLUTION HERE
im_reconstruct_translation = np.zeros(im.shape)
estimated_translations = np.zeros((N, 2))

compare_reconstruction(im, im_reconstruct_translation, 'Rotation + translation')